In [ ]:
import os

MAX_REASONABLE_SIZES = {
    "image/gif": 25 * 1024 * 1024,       # GIFs > 25MB are usually bloated converted videos
    "image/png": 20 * 1024 * 1024,       # Standard screenshots rarely exceed 20MB
    "application/pdf": 50 * 1024 * 1024, # PDFs > 50MB often conceal embedded media
}

def inspect_media_file(filepath):
    size_bytes = os.path.getsize(filepath)
    size_mb = size_bytes / (1024 * 1024)

    with open(filepath, "rb") as f:
        head = f.read(64)

    # Native container signatures
    detected = "Unknown"
    if head.startswith(b"GIF87a") or head.startswith(b"GIF89a"):
        detected = "GIF animation"
    elif b"ftyp" in head[4:16]:
        detected = "MP4/ISO container"
    elif head.startswith(b"\x1a\x45\xdf\xa3"):
        detected = "EBML (MKV/WebM video)"
    elif head.startswith(b"RIFF") and head[8:12] == b"WEBP":
        detected = "WebP (possibly animated)"

    print(f"File:     {os.path.basename(filepath)}")
    print(f"Size:     {size_mb:.2f} MB ({size_bytes:,} bytes)")
    print(f"Format:   {detected}")

    # Warning thresholds
    if size_mb > 50 and detected in ("GIF animation", "WebP (possibly animated)"):
        print("[WARNING] Abnormal container weight: High likelihood of uncompressed video or embedded payload.")
    elif size_mb > 100:
        print("[WARNING] Heavy media object (>100MB). Skip in-memory pixel arrays to prevent OOM errors.")

In [4]:

inspect_media_file("test_files/from_web/857f75db04db84340d4efa53061ca26fc7c7e6f65f80899c57f8d65b4c9d5678")

File:     857f75db04db84340d4efa53061ca26fc7c7e6f65f80899c57f8d65b4c9d5678
Size:     229.92 MB (241,083,862 bytes)
Format:   WebP (possibly animated)
[WARNING] Abnormal container weight: High likelihood of uncompressed video or embedded payload.


In [1]:
import os
import struct

TARGET_WEBP = "test_files/from_web/85/7f/857f75db04db84340d4efa53061ca26fc7c7e6f65f80899c57f8d65b4c9d5678"

if not os.path.exists(TARGET_WEBP):
    # Fallback to search if nested differently
    matches = [p for p in os.popen(f"find test_files -name '*857f*'").read().split() if not p.endswith(".png")]
    TARGET_WEBP = matches[0] if matches else TARGET_WEBP

file_size = os.path.getsize(TARGET_WEBP)
print(f"Inspecting: {os.path.basename(TARGET_WEBP)} ({file_size:,} bytes)")
print("-" * 65)

with open(TARGET_WEBP, "rb") as f:
    header = f.read(12)
    if not (header.startswith(b"RIFF") and header[8:12] == b"WEBP"):
        print("[!] Not a valid RIFF/WEBP container!")
    else:
        declared_riff_size = struct.unpack("<I", header[4:8])[0]
        expected_total_size = declared_riff_size + 8
        
        print(f"Declared RIFF size:   {declared_riff_size:,} bytes")
        print(f"Expected file size:   {expected_total_size:,} bytes")
        print(f"Actual file size:     {file_size:,} bytes")
        
        # 1. Check for appended overlay data past RIFF boundary
        trailing = file_size - expected_total_size
        if trailing > 0:
            print(f"\n[!] ALERT: {trailing:,} trailing bytes appended past the RIFF container!")
            f.seek(expected_total_size)
            overlay = f.read(64)
            print(f"    Trailing hex preview: {overlay[:32].hex(' ')}")
            print(f"    Trailing ASCII:       {''.join(chr(b) if 32 <= b < 127 else '.' for b in overlay[:32])}")
            if overlay.startswith(b"PK\x03\x04"):
                print("    --> Found nested ZIP archive in overlay!")
        elif trailing < 0:
            print(f"\n[!] File is truncated (missing {abs(trailing):,} bytes)")
        else:
            print("[+] Container Boundary: Clean (no appended trailing overlay data).")

        # 2. Walk RIFF Chunks without loading image frames into memory
        print("\n--- RIFF Chunk Map ---")
        offset = 12
        chunk_counts = {}
        anmf_frames = 0
        
        while offset < file_size:
            f.seek(offset)
            chunk_hdr = f.read(8)
            if len(chunk_hdr) < 8:
                break
                
            fourcc = chunk_hdr[:4].decode("ascii", errors="replace")
            chunk_size = struct.unpack("<I", chunk_hdr[4:8])[0]
            
            chunk_counts[fourcc] = chunk_counts.get(fourcc, 0) + 1
            if fourcc == "ANMF":
                anmf_frames += 1
                
            # Suspicious metadata/unknown chunk check
            if fourcc in ("EXIF", "XMP ", "ICCP", "SKIP"):
                chunk_data = f.read(min(chunk_size, 64))
                print(f"  * [{fourcc}] Metadata chunk ({chunk_size:,} bytes)")
                print(f"    Preview: {''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk_data[:32])}")
            
            # RIFF chunks are padded to 2-byte alignment
            offset += 8 + chunk_size + (chunk_size % 2)

        print("\n--- Summary ---")
        for cc, count in chunk_counts.items():
            print(f"  Chunk '{cc}': {count:,} occurrence(s)")
            
        if anmf_frames > 0:
            print(f"\nResult: Animated WebP containing {anmf_frames:,} frames.")

Inspecting: 857f75db04db84340d4efa53061ca26fc7c7e6f65f80899c57f8d65b4c9d5678 (241,083,862 bytes)
-----------------------------------------------------------------
Declared RIFF size:   241,083,854 bytes
Expected file size:   241,083,862 bytes
Actual file size:     241,083,862 bytes
[+] Container Boundary: Clean (no appended trailing overlay data).

--- RIFF Chunk Map ---
  * [EXIF] Metadata chunk (36 bytes)
    Preview: II*.......1...............ezgif.

--- Summary ---
  Chunk 'VP8X': 1 occurrence(s)
  Chunk 'ANIM': 1 occurrence(s)
  Chunk 'ANMF': 12,267 occurrence(s)
  Chunk 'EXIF': 1 occurrence(s)

Result: Animated WebP containing 12,267 frames.
